# E2 — gpt-oss-20b(경제 사전학습) 금융 NER 파인튜닝·채점 (Colab, 팀원용)

**무엇**: park_gyutae 님이 경제 문장 50만으로 사전학습한 gpt-oss-20b(LoRA)에, 금융 라벨(8태그)로 **"문장 → 개체 목록을 글로 답하기"** 를 학습시키고 dev에서 **span F1**을 잰다.
- 답 형식: `개체=태그 | 개체=태그` (없으면 `없음`). 같은 글자가 문장에 여러 번 나오면 두 번째부터 `개체@2=태그`.
- 채점: 답을 원문 글자 위치로 되돌려 (시작·끝·태그)가 정답과 정확히 같으면 맞음. 문장에 없는 개체를 쓰면 위치를 못 찾아 오답(fp)으로 센다.
- 이 변환은 Claude 표본 1만 문장(개체 22,718개)에서 정답 → 답 글 → 위치로 되돌렸을 때 **전부 일치**하는 것을 확인했다.

## 데이터·채점 (F1 노트북과 같음)
- 폴더 `NER_라벨_통합_v1.2/finetune` 의 `data/`: 학습 28만 · dev 1만 · test 1만 · **test_2000**(모든 모델 공통 비교 2천 문장). GUIDE v1.2, 원본 파일 단위 분할.
- 채점은 `fin_score.py`(모든 금융 노트북 공통): (시작·끝·태그) 정확히 같아야 맞음, micro F1 + 고정 8태그 macro F1, 문장별 오답 기록.
- 생성이 느려 test 1만 대신 **test_2000** 으로 채점 → 인코더 모델들도 같은 2천 문장 점수를 같이 낸다.

## 런타임
| 항목 | 고를 것 |
|---|---|
| GPU | **H100** 또는 **G4(96GB)** 또는 **A100 80GB** (70GB 이상 필수, 1번 셀이 확인) |
| RAM | 표준 |

## 예상 자원 (⚠ 추정 — 5번 셀이 `[속도]` 줄로 실제 값을 출력)
| 항목 | 값 |
|---|---|
| 준비 | 모델 13GB 다운로드 + 사전학습 LoRA 합치기 10~20분 |
| 학습 | 1만 문장 1에폭 ≈ 313스텝(배치 8×4), 대략 수십 분 |
| 채점(생성) | 답을 한 글자씩 써서 느림. dev 500문장에 수십 분 — `EVAL_N` 으로 조절 |

## 저장 방식
- 학습 결과·체크포인트는 Colab 로컬(`/content/out`), 체크포인트는 저장할 때마다 이 폴더 `ckpt/` 에도 복사 → 끊기면 1~5번 다시 실행하면 이어서 학습.
- 끝나면 6번 셀이 결과(새 LoRA·점수 json·dev 예측)만 이 폴더 `결과_이름/` 에 올리고, 7번 셀이 zip을 내 PC로 내려받은 뒤 **런타임 반납(세션 종료)**.

## 순서
위에서부터 모두 실행. 4번 셀에서 TRAIN·DEV·NAME 만 정하면 된다. 결과(6번 출력)의 dev F1·태그별 F1·학습 시간을 팀 채널에 공유.


In [ ]:
# 1. GPU 확인 (70GB 이상)
import torch, subprocess
assert torch.cuda.is_available(), 'GPU 런타임이 아닙니다'
p = torch.cuda.get_device_properties(0); print(p.name, f'{p.total_memory/2**30:.0f}GB')
assert p.total_memory > 70 * 2**30, 'GPU 메모리 70GB 미만 → H100 / G4 / A100 80GB 로 바꾸세요'

In [ ]:
# 2. 드라이브 연결 · 폴더 찾기 (NER_라벨_통합_v1.2/finetune — F1 노트북과 같은 폴더·데이터·채점)
from google.colab import drive
drive.mount('/content/drive')
import os, glob, json
ROOT = '/content/drive/MyDrive/NER_라벨_통합_v1.2/finetune'   # 다르면 여기만 수정
if not os.path.exists(f'{ROOT}/ft_gptoss_ner.py'):
    found = [os.path.dirname(p) for p in glob.glob('/content/drive/MyDrive/**/ft_gptoss_ner.py', recursive=True)
             + glob.glob('/content/drive/Shareddrives/*/**/ft_gptoss_ner.py', recursive=True)]
    print('찾은 위치:', found); assert found, 'ft_gptoss_ner.py 를 못 찾음'
    ROOT = found[0]
print('ROOT =', ROOT)
DAPT = f'{ROOT}/models/09_gptoss_dapt-lora'
assert os.path.getsize(f'{DAPT}/adapter_model.safetensors') == 149_843_368, '사전학습 LoRA 파일이 덜 올라왔습니다'
print('사전학습 LoRA OK · 데이터', sorted(os.listdir(f'{ROOT}/data')))

In [ ]:
# 3. 라이브러리
!pip install -q -U transformers peft accelerate
import transformers, peft; print('transformers', transformers.__version__, '· peft', peft.__version__)

In [ ]:
# 4. 데이터 (기본: F1 노트북과 같은 학습 28만 · dev 1만 · 공통 test_2000)
TRAIN = f'{ROOT}/data/train.jsonl'
DEV   = f'{ROOT}/data/dev.jsonl'
TEST2K = f'{ROOT}/data/test_2000.jsonl'   # 모든 모델 공통 비교 2,000문장
NAME  = 'F09_gptoss-dapt_gen'
TRAIN_N = 0      # 0 = 28만 전부(학습 약 3시간 이상 추정). 빠른 시험은 50000 등
EVAL_N  = 500    # 학습 중 확인용 dev 문장 수(생성이 느림)
EXTRA = f'{ROOT}/data/crawl350_fable.jsonl'   # Fable 라벨 크롤링 문장(준비되면 자동 채점)
TAGS = {'FIN_ORG','ORG','FIN_PRODUCT','MONEY','RATE','DATE','INDICATOR','POLICY'}
def check(p):
    rows, bad = [], 0
    for l in open(p, encoding='utf-8'):
        if not l.strip(): continue
        r = json.loads(l); rows.append(r)
        if any(not (0 <= s < e <= len(r['text'])) or t not in TAGS for s, e, t in r['spans']): bad += 1
    return rows, bad
tr, b1 = check(TRAIN); dv, b2 = check(DEV)
print(f'학습 {len(tr):,}(형식 오류 {b1}) · dev {len(dv):,}(형식 오류 {b2}) · 학습∩dev 같은 문장 {len({r["text"] for r in tr} & {r["text"] for r in dv})} · 추가 채점 {os.path.exists(EXTRA)}')
assert b1 == 0 and b2 == 0

In [ ]:
# 5. 학습 + 채점(dev 앞 EVAL_N · test_2000 · 추가 파일) — 체크포인트는 드라이브 results/_ckpt 에도 복사, 끊기면 1~5번 다시
OUT = '/content/out'
EX = f'--eval_file "{EXTRA}"' if os.path.exists(EXTRA) else ''
!cd /content && HF_HOME=/content/hf python "{ROOT}/ft_gptoss_ner.py" --train "{TRAIN}" --dev "{DEV}" --test2k "{TEST2K}" --dapt "{DAPT}" --out_root {OUT} --mirror "{ROOT}/results/_ckpt" --name {NAME} --train_n {TRAIN_N} --eval_n {EVAL_N} {EX}

In [ ]:
# 6. 결과 확인 → 드라이브 results/NAME 에 올리고 zip
import shutil
r = json.load(open(f'{OUT}/results/{NAME}.json', encoding='utf-8'))
for k, v in r['evals'].items():
    s = v['score']; print(f"{k:>9} {v['n']}문장: F1 {s['ALL']['f1']} (P {s['ALL']['precision']} R {s['ALL']['recall']}) · macro {s['MACRO_F1']} · 오답 {v['error_types']} · 위치 못 찾은 답 {v['unparsed_or_not_found']}")
print(f"학습 {r['train_minutes']}분 · 채점 {r['gen_minutes']}분 · 저장 확인 {r['verify_ok']}")
UP = f'{ROOT}/results/{NAME}'; os.makedirs(UP, exist_ok=True)
shutil.copytree(f'{OUT}/models/{NAME}', f'{UP}/model', dirs_exist_ok=True); shutil.copy(f'{OUT}/results/{NAME}.json', UP)
for f in glob.glob(f'{OUT}/preds/{NAME}_*.jsonl'): shutil.copy(f, UP)
same = all(os.path.getsize(f'{OUT}/models/{NAME}/{f}') == os.path.getsize(f'{UP}/model/{f}') for f in os.listdir(f'{OUT}/models/{NAME}'))
shutil.make_archive(f'/content/{NAME}', 'zip', UP)
print('드라이브 업로드', UP, '· 크기 일치', same)
VERIFY_OK = bool(r.get('verify_ok')) and same

In [ ]:
# 7. 내 PC로 내려받기 + 런타임 반납(세션 종료) — 저장 확인·업로드가 통과했을 때만
import time
from google.colab import files, runtime
if VERIFY_OK:
    files.download(f'/content/{NAME}.zip'); print('내려받기 시작 — 2분 뒤 런타임 반납'); time.sleep(120); runtime.unassign()
else:
    print('저장 확인 또는 업로드 실패 → 반납하지 않음. 출력을 공유해 주세요')